# Задача 2. K-th minheap

## Ход решения

Строю minheap из первых `k` элементов. Если очередное число больше корня, заменяю корень этим числом и просеиваю его вниз. Тогда в куче остаются `k` наибольших значений среди уже просмотренных. Равное корню значение можно пропустить, поскольку замена сохранила бы тот же набор с учётом повторов. Когда весь список просмотрен, корень даёт k-й наибольший элемент.

В ручной версии строю кучу снизу вверх и восстанавливаю порядок спуском по меньшему ребёнку. В библиотечной версии те же операции делают `heapify` и `heapreplace`. Построение занимает $O(k)$, а каждая замена — до $O(\log k)$, поэтому общее время составляет $O(n+(n-k)\log k)$. Дополнительная память — $O(k)$ под отдельную кучу.

## Реализация без heapq

In [1]:
def _sift_down_min(arr, root, stop):
    """Восстанавливает minheap под root в пределах arr[0:stop]."""
    while True:
        left = 2 * root + 1
        if left >= stop:
            return

        right = left + 1
        child = left
        if right < stop and arr[right] < arr[left]:
            child = right

        if not arr[child] < arr[root]:
            return

        arr[root], arr[child] = arr[child], arr[root]
        root = child


def makeheap(arr):
    """Строит minheap просеиваниями вниз в O(n) и возвращает исходный список."""
    stop = len(arr)
    for root in range(stop // 2 - 1, -1, -1):
        _sift_down_min(arr, root, stop)
    return arr


def _validate_rank(nums, k):
    if isinstance(k, bool) or not isinstance(k, int):
        raise TypeError("k должно быть целым числом")
    if not 1 <= k <= len(nums):
        raise ValueError("k должно находиться в диапазоне от 1 до len(nums)")


def kth_largest(nums, k):
    """Возвращает k-й наибольший элемент с помощью собственной minheap."""
    _validate_rank(nums, k)
    heap = makeheap(nums[:k])

    for index in range(k, len(nums)):
        candidate = nums[index]
        if heap[0] < candidate:
            heap[0] = candidate
            _sift_down_min(heap, 0, k)
    return heap[0]

## Реализация с heapq

In [2]:
def kth_largest_heapq(nums, k):
    """Возвращает k-й наибольший элемент с помощью стандартной heapq."""
    from heapq import heapify, heapreplace

    _validate_rank(nums, k)
    heap = nums[:k]
    heapify(heap)

    for index in range(k, len(nums)):
        candidate = nums[index]
        if heap[0] < candidate:
            heapreplace(heap, candidate)
    return heap[0]

## Пошаговый пример

Возьмём `nums = [3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2]` и `k = 4`. Два последних значения показывают обе причины пропуска. В собственной версии первые четыре элемента образуют кучу за два просеивания.

| Шаг построения | Состояния после каждого обмена |
| --- | --- |
| Исходный срез | `[3, 2, 3, 1]` |
| Просеивание из `1` | `[3, 1, 3, 2]` |
| Просеивание из `0` | `[1, 3, 3, 2]` → `[1, 2, 3, 3]` |

Библиотечная версия после `heapify` получает `[1, 2, 3, 3]`. Ниже указан каждый дальнейший шаг. При равных детях способы просеивания могут выбирать разные ветви, поэтому расположение внутри куч может отличаться при совпадающих значениях и кратностях.

| Индекс | `x` | Корень до шага | Действие в собственной реализации | Собственная minheap после шага | `heapq` после шага |
| ---: | ---: | ---: | --- | --- | --- |
| 4 | 2 | 1 | Заменяем корень, порядок уже соблюдён | `[2, 2, 3, 3]` | `[2, 2, 3, 3]` |
| 5 | 4 | 2 | Заменяем корень, обмены 0 ↔ 1 и 1 ↔ 3 | `[2, 3, 3, 4]` | `[2, 3, 3, 4]` |
| 6 | 5 | 2 | Заменяем корень, обмены 0 ↔ 1 и 1 ↔ 3 | `[3, 4, 3, 5]` | `[3, 3, 5, 4]` |
| 7 | 5 | 3 | Заменяем корень, обмен 0 ↔ 2 | `[3, 4, 5, 5]` | `[3, 4, 5, 5]` |
| 8 | 6 | 3 | Заменяем корень, обмены 0 ↔ 1 и 1 ↔ 3 | `[4, 5, 5, 6]` | `[4, 5, 5, 6]` |
| 9 | 4 | 4 | Пропускаем, равен корню | `[4, 5, 5, 6]` | `[4, 5, 5, 6]` |
| 10 | -2 | 4 | Пропускаем, меньше корня | `[4, 5, 5, 6]` | `[4, 5, 5, 6]` |

После прохода в куче находятся `6, 5, 5, 4`. Корень `4` — четвёртый по величине, поскольку обе пятёрки занимают отдельные места.

In [3]:
cases = (
    ([3, 2, 1, 5, 6, 4], 2, 5),
    ([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4),
    ([3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2], 4, 4),
)
for nums, k, expected in cases:
    own = kth_largest(nums, k)
    library = kth_largest_heapq(nums, k)
    assert own == library == expected
    print(f"nums = {nums}, k = {k} → своя куча: {own}, heapq: {library}")

nums = [3, 2, 1, 5, 6, 4], k = 2 → своя куча: 5, heapq: 5
nums = [3, 2, 3, 1, 2, 4, 5, 5, 6], k = 4 → своя куча: 4, heapq: 4
nums = [3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2], k = 4 → своя куча: 4, heapq: 4


## Тесты

У обеих версий проверяю ответы из условия, граничные ранги, все ранги на малых списках и сохранность `nums`. Пустой вход, нулевой, отрицательный и слишком большой ранг должны вызывать `ValueError`. Дробный, строковый и логический ранг проверяю на `TypeError`.

Использую типичные входы, граничные случаи и строки как пример других сравнимых значений. Перебираю все 1 093 списка длины от 0 до 6 из значений −1, 0 и 1. Дополнительно генерирую 2 000 списков с фиксированным зерном, повторами и отрицательными числами. Отдельно проверяю возрастающий, убывающий и полностью одинаковый списки длины 100 000.

In [4]:
from itertools import product
from random import Random
import sys
import unittest


RANK_FINDERS = (kth_largest, kth_largest_heapq)


EXAMPLES = (
    [],
    [42],
    [2, 1],
    [1, 2],
    list(range(15)),
    list(range(15, -1, -1)),
    [5] * 17,
    [0, -3, 2, -3, 0, 2, -1],
    [7, 2, 9, 4, 1, 8, 5, 3],
    [3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2],
    ["б", "в", "а", "а"],
)


def small_inputs():
    for size in range(7):
        for values in product((-1, 0, 1), repeat=size):
            yield list(values)


def random_inputs():
    rng = Random(314159)
    for _ in range(2_000):
        size = rng.randrange(129)
        yield [rng.randint(-30, 30) for _ in range(size)]


def large_inputs():
    size = 100_000
    return (
        ("ascending", list(range(size))),
        ("descending", list(range(size - 1, -1, -1))),
        ("equal", [5] * size),
    )


class KthLargestTests(unittest.TestCase):
    def check_rank(self, values, k, expected):
        for find in RANK_FINDERS:
            with self.subTest(method=find.__name__, n=len(values), k=k):
                nums = values.copy()
                self.assertEqual(find(nums, k), expected)
                self.assertEqual(nums, values)

    def check_all_ranks(self, values):
        for k, expected in enumerate(sorted(values, reverse=True), start=1):
            self.check_rank(values, k, expected)

    def test_statement_and_trace_examples(self):
        self.check_rank([3, 2, 1, 5, 6, 4], 2, 5)
        self.check_rank([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4)
        self.check_rank([3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2], 4, 4)

    def test_edge_and_typical_ranks(self):
        for values in EXAMPLES:
            with self.subTest(values=values):
                self.check_all_ranks(values)

    def test_invalid_ranks(self):
        for find in RANK_FINDERS:
            for values, k in (([], 1), ([2, 1], -1), ([2, 1], 0), ([2, 1], 3)):
                with self.subTest(method=find.__name__, values=values, k=k):
                    with self.assertRaises(ValueError):
                        find(values, k)
            for k in (1.0, "1", True, None):
                with self.subTest(method=find.__name__, k=k):
                    with self.assertRaises(TypeError):
                        find([2, 1], k)

    def test_exhaustive_small_inputs(self):
        for values in small_inputs():
            self.check_all_ranks(values)

    def test_seeded_random_inputs(self):
        rng = Random(271828)
        for case, values in enumerate(random_inputs()):
            if values:
                with self.subTest(case=case):
                    expected = sorted(values, reverse=True)
                    for k in {1, len(values), rng.randint(1, len(values))}:
                        self.check_rank(values, k, expected[k - 1])

    def test_large_patterns(self):
        for name, values in large_inputs():
            with self.subTest(pattern=name):
                expected = sorted(values, reverse=True)
                for k in (1, len(values) // 2, len(values)):
                    self.check_rank(values, k, expected[k - 1])

In [5]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(KthLargestTests)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful()

test_edge_and_typical_ranks (__main__.KthLargestTests.test_edge_and_typical_ranks) ... ok
test_exhaustive_small_inputs (__main__.KthLargestTests.test_exhaustive_small_inputs) ... ok
test_invalid_ranks (__main__.KthLargestTests.test_invalid_ranks) ... ok
test_large_patterns (__main__.KthLargestTests.test_large_patterns) ... ok
test_seeded_random_inputs (__main__.KthLargestTests.test_seeded_random_inputs) ... ok
test_statement_and_trace_examples (__main__.KthLargestTests.test_statement_and_trace_examples) ... ok

----------------------------------------------------------------------
Ran 6 tests in 1.414s

OK
